In [2]:
import pandas as pd
import numpy as np

import torch
from torch import nn
from torch import optim
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import mean_squared_error, r2_score

In [3]:
FILE_PATH_TRAIN = "../data/split/train/train.csv"
df_train = pd.read_csv(FILE_PATH_TRAIN)
FILE_PATH_TEST = "../data/split/test/test.csv"
df_test = pd.read_csv(FILE_PATH_TEST)

Hồ Chí Minh chia ra thành 3 khu vực (theo Quyết định 79/2024/QĐ-UBND bảng giá đất TP.HCM):
- Khu vực 1: Quận 1, Quận 3, Quận 4, Quận 5, Quận 6, Quận 10, Quận 11, quận Bình Thạnh, quận Phú Nhuận.
- Khu vực 2: Quận 7, Quận 8, Quận 12, quận Tân Bình, quận Tân Phú, quận Bình Tân, quận Gò Vấp, thành phố Thủ Đức.
- Khu vực 3: Bình Chánh, huyện Hóc Môn, huyện Củ Chi (Không có dữ liệu), huyện Nhà Bè, huyện Cần Giờ (không có dữ liệu).

In [4]:
khu_vuc_1 = [
    'quận 1', 'quận 3', 'quận 4', 'quận 5', 'quận 6',
    'quận 10', 'quận 11', 'bình thạnh', 'phú nhuận'
]

khu_vuc_2 = [
    'quận 7', 'quận 8', 'quận 12', 'tân bình', 'tân phú',
    'bình tân', 'gò vấp', 'thủ đức', 'quận 9', 'quận 2'
]

khu_vuc_3 = [
    'bình chánh', 'nhà bè', 'cần giờ'
]


In [5]:
region_mapping = {}
region_mapping.update({quan: 'Khu vực 1' for quan in khu_vuc_1})
region_mapping.update({quan: 'Khu vực 2' for quan in khu_vuc_2})
region_mapping.update({quan: 'Khu vực 3' for quan in khu_vuc_3})

In [6]:
df_train['region'] = df_train['address'].map(region_mapping)
df_test['region'] = df_test['address'].map(region_mapping)

In [7]:
def assign_region(addr):
    if addr in khu_vuc_1:
        return 'Khu vực 1'
    if addr in khu_vuc_2:
        return 'Khu vực 2'
    if addr in khu_vuc_3:
        return 'Khu vực 3'
    return 'Khác'

df_train['region'] = df_train['address'].apply(assign_region)
df_test['region'] = df_test['address'].apply(assign_region)


In [8]:
region_map = {
    'Khu vực 1': 1,
    'Khu vực 2': 2,
    'Khu vực 3': 3,
    'Khác': 0
}

df_train['region_encoded'] = df_train['region'].map(region_map)
df_test['region_encoded'] = df_test['region'].map(region_map)

In [9]:
X_train = df_train[['area', 'bedrooms', 'bathrooms', 'region_encoded']]
y_train = df_train['price']
X_test = df_test[['area', 'bedrooms', 'bathrooms', 'region_encoded']]
y_test = df_test['price']

In [10]:
df_train.head()

,price,area,bedrooms,bathrooms,address,region,region_encoded
0,4.25,75.0,2,2,quận 9,Khu vực 2,2
1,9.70,75.2,2,2,quận 2,Khu vực 2,2
2,7.50,80.0,6,5,tân phú,Khu vực 2,2
3,36.00,217.0,3,4,quận 2,Khu vực 2,2
4,3.50,81.3,3,2,quận 9,Khu vực 2,2


In [11]:
scaler_X = StandardScaler()
X_train_s = scaler_X.fit_transform(X_train)
X_test_s = scaler_X.transform(X_test)



In [12]:
X_train_t = torch.tensor(X_train_s, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32) 
X_test_t = torch.tensor(X_test_s, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)

In [13]:
model = nn.Linear(X_train_t.shape[1], 1)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001) 

In [14]:
epochs = 10000 

train_losses = [] 

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    y_pred_train = model(X_train_t)
    loss = criterion(y_pred_train, y_train_t)
    loss.backward()
    optimizer.step()
    
    train_losses.append(loss.item())
    
    if (epoch+1) % 1000 == 0:
        print(f"Epoch [{epoch+1}/{epochs}] - Loss: {loss.item():.6f}")

d:\HCMUS\HK8\ML Introduction\lab\lab1\INTROML\.venv\Lib\site-packages\torch\nn\modules\loss.py:634: UserWarning: Using a target size (torch.Size([1600])) that is different to the input size (torch.Size([1600, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch [1000/10000] - Loss: 140.425293
Epoch [2000/10000] - Loss: 128.010056
Epoch [3000/10000] - Loss: 117.704185
Epoch [4000/10000] - Loss: 109.264412
Epoch [5000/10000] - Loss: 102.525154
Epoch [6000/10000] - Loss: 97.356064
Epoch [7000/10000] - Loss: 93.628151
Epoch [8000/10000] - Loss: 91.185341
Epoch [9000/10000] - Loss: 89.816467
Epoch [10000/10000] - Loss: 89.231651


In [ ]:
model.eval()
with torch.no_grad():
    y_pred_t = model(X_test_t).squeeze().cpu().numpy()
    # Không scale y -> dùng trực tiếp
    y_pred = y_pred_t
    y_test_orig = y_test.values

In [17]:
r2 = r2_score(y_test_orig, y_pred)
rmse = np.sqrt(mean_squared_error(y_test_orig, y_pred))
print(f"R2: {r2:.4f}, RMSE: {rmse:.4f}")

R2: -43.2383, RMSE: 76.0061
